# 2023E 黄河水沙监测：可复现实验

证据等级：题面/附件字段为 A；代码复算结果为 B；解释和建议为 C。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "请从仓库内运行 Notebook"


## 1. 原始附件校验与数据字典

运行下载/提取脚本后读取官方附件，不创建模拟数据。

In [ ]:
from cumcm_lens.cases.yellow_river import locate_attachments, load_monitoring
files = locate_attachments(ROOT / "data/raw")
monitoring, quality = load_monitoring(files["附件1.xlsx"])
quality


## 2. 缺失审计

含沙量缺失不会被前向填充；只在时间外推测试后由最终模型补齐。

In [ ]:
monitoring[["water_level_m", "discharge_m3s", "sediment_kgm3"]].isna().mean()


## 3. 三模型比较与泄漏检查

In [ ]:
from cumcm_lens.cases.yellow_river import train_sediment_models
modeled, detail, config = train_sediment_models(monitoring, seed=42)
pd.DataFrame(detail["metrics"])


In [ ]:
pd.DataFrame(detail['audits'])

## 4. 年度水沙通量

使用不规则时距梯形积分，超过48小时的长缺口不被直接跨越。

In [ ]:
from cumcm_lens.cases.yellow_river import integrate_annual_flux
annual = integrate_annual_flux(modeled)
annual


## 5. 月尺度规律、回测与两年预测

In [ ]:
from cumcm_lens.cases.yellow_river import monthly_series, forecast_24_months
monthly = monthly_series(modeled)
forecast, comparison = forecast_24_months(monthly)
pd.DataFrame(comparison)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(monthly["timestamp"], monthly["discharge_m3s"])
axes[1].plot(monthly["timestamp"], monthly["sediment_kgm3"])
plt.show()


## 6. 断面变化与采样计划

In [ ]:
from cumcm_lens.cases.yellow_river import cross_section_summary, sampling_plan
display(cross_section_summary(files["附件2.xlsx"]))
display(sampling_plan(forecast))


## 7. 结论边界

测试期 R² 较低说明跨年份泛化有限。年度输沙量和未来预测必须连同误差、缺失率与模型假设一起引用。